# Phase 1 — Legal Language Model (from scratch) · First Submission
**CS781 Capstone · Kaggle GPU notebook**

This notebook takes you end-to-end for your **first Phase-1 submission**:

1. Log in to Hugging Face and load the gated `Exploration-Lab/CS-781-Capstone` dataset.
2. Train **your own Assignment-1 byte-level BPE tokenizer** on the legal corpus.
3. Build a decoder-only Transformer using **your own Assignment-3 code** (`model_arch.py`, `attention.py`, `rope.py`), scaled to the 50M–300M range required by Phase 1.
4. Train it from scratch on the legal corpus.
5. Generate a 128-word continuation for every official test prompt.
6. Write `prediction.json` and package `submission.zip` (json at the ZIP root) — ready to upload to Codabench.

---
### How to run on Kaggle
1. Notebook **Settings → Accelerator → GPU** (T4/P100), and **Internet → On**.
2. Add your HF token as a **Secret** named `HF_TOKEN` (Add-ons → Secrets), *or* just paste it when prompted.
3. Keep `SMOKE_TEST = True` for the first run — it finishes in a few minutes and proves the whole pipeline + submission file are correct. Then set `SMOKE_TEST = False` for a real training run.

In [ ]:
# 1 · Install dependencies (torch is already on Kaggle). Requires Internet = ON.
!pip -q install -U "huggingface_hub>=0.23" "datasets>=2.19" ijson

In [ ]:
# 2 · Configuration  --------------------------------------------------------
# Flip this to False for a real training run once the pipeline works.
SMOKE_TEST = True

CONFIG = dict(
    hf_repo   = "Exploration-Lab/CS-781-Capstone",
    phase_dir = "Phase_1_Pretraining",
    sources   = ["acts", "high_courts", "supreme_court", "tribunals"],

    # ---- tokenizer (your A1 byte-level BPE, trained on legal text) ----
    vocab_size     = 8000,          # A3 used 8000; raise later (16000/32000) as an experiment
    tok_train_chars= 12_000_000,    # amount of legal text used to TRAIN the tokenizer

    # ---- training data budget ----
    train_tokens = 20_000_000,      # tokens used for training (cap; raise if you have compute)
    val_tokens   = 1_000_000,

    # ---- model (your A3 DecoderLM), scaled to 50M-300M ----
    d_model      = 768,
    n_layers     = 10,
    n_heads      = 12,
    n_kv_heads   = 12,              # =n_heads -> MHA. Set <n_heads for GQA (a later architecture)
    d_ff         = 3072,
    max_seq_len  = 1024,
    dropout      = 0.1,
    pos_emb_type = "sinusoidal",    # "sinusoidal" or "rope" (both are your A3 code)

    # ---- training ----
    batch_size   = 8,
    grad_accum   = 4,               # effective batch = batch_size * grad_accum
    lr           = 3e-4,
    weight_decay = 0.1,
    warmup_steps = 200,
    max_steps    = 4000,
    eval_every   = 250,
    grad_clip    = 1.0,

    # ---- generation ----
    gen_words      = 128,           # Phase-1 asks for a 128-word continuation
    max_new_tokens = 320,           # generate a bit more, then trim to 128 words
    temperature    = 0.8,
    top_k          = 50,
    top_p          = 0.95,

    seed    = 1337,
    out_dir = "/kaggle/working",
)

if SMOKE_TEST:
    CONFIG.update(
        vocab_size=2000, tok_train_chars=1_000_000,
        train_tokens=300_000, val_tokens=50_000,
        d_model=256, n_layers=4, n_heads=4, n_kv_heads=4, d_ff=1024,
        max_seq_len=256, batch_size=8, grad_accum=1,
        warmup_steps=20, max_steps=120, eval_every=40, max_new_tokens=160,
    )

import os, random, math, json, time, pickle, zipfile
import torch, torch.nn.functional as F
random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CONFIG["out_dir"], exist_ok=True)
print("Device:", device, "| SMOKE_TEST:", SMOKE_TEST)

In [ ]:
# 3 · Hugging Face login (needed because the dataset is gated) -------------
# Accept the dataset terms once at:
#   https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone
from huggingface_hub import login
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets.")
except Exception:
    pass
if not token:
    import getpass
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Enter your Hugging Face token: ")
login(token=token)
print("Logged in to Hugging Face.")

## 4 · Your Assignment-3 code (pasted verbatim)
The next four cells write **your own A3 implementations** to disk so we can import them. This is your code, unchanged — keep it that way for the viva.

In [ ]:
import os
os.makedirs('a3src', exist_ok=True)

In [ ]:
%%writefile a3src/rope.py
# TODO 3.1: rotary positional embeddings from scratch (split-half convention).
import torch


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([angles, angles], dim=-1)                               # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
%%writefile a3src/attention.py
# TODO 3.2: MHA / GQA / MQA attention with optional RoPE applied to q and k.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0,
                 use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        if attention_scale is None:
            self.attention_scale = 1.0 / math.sqrt(self.d_head)
        else:
            self.attention_scale = float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.attention_scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, self.n_heads * self.d_head)
        return self.resid_dropout(self.out_proj(out))

In [ ]:
%%writefile a3src/model_arch.py
# TODO 3.2: pass use_rope through the decoder; only build sinusoidal PE when requested.
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

In [ ]:
%%writefile a3src/a1_tokenizer.py
"""Assignment-1 byte-level BPE tokenizer, retrained on the Assignment-3 corpus.

Byte-level BPE (A-1 Q5 spec): byte symbols so no <unk> is needed, reserved tokens
<pad> <unk> <s> </s> at ids 0..3, each merge takes the most frequent adjacent pair
(ties broken by the lexicographically smallest merged bytes), and merges never cross
a pre-token (word / whitespace) boundary so decode() reconstructs the text exactly.
"""
import re
from collections import Counter

RESERVED = ("<pad>", "<unk>", "<s>", "</s>")
BASE = len(RESERVED)
SPLIT = re.compile(r"\S+|\s+")


def merge_sequence(ids, pair, new_id):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)


class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.ranks = {}
        self.itos = {}
        self.vocab_size = 0
        self.cache = {}

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))

        counts = Counter(SPLIT.findall(text))
        corpus = Counter()
        for word, freq in counts.items():
            ids = tuple(BASE + byte for byte in word.encode("utf-8"))
            corpus[ids] += freq

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges:
            pair_freq = Counter()
            for ids, freq in corpus.items():
                for j in range(len(ids) - 1):
                    pair_freq[(ids[j], ids[j + 1])] += freq
            if not pair_freq:
                break
            top = max(pair_freq.values())
            if top < 2:
                break
            best_pairs = [p for p, f in pair_freq.items() if f == top]
            pair = min(best_pairs, key=lambda p: self.itos[p[0]] + self.itos[p[1]])

            self.itos[next_id] = self.itos[pair[0]] + self.itos[pair[1]]
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)

            new_corpus = Counter()
            for ids, freq in corpus.items():
                new_corpus[merge_sequence(ids, pair, next_id)] += freq
            corpus = new_corpus
            next_id += 1

        self.vocab_size = BASE + 256 + len(self.merges)

    def encode_chunk(self, raw_bytes):
        if raw_bytes in self.cache:
            return self.cache[raw_bytes]
        ids = [BASE + byte for byte in raw_bytes]
        while len(ids) >= 2:
            candidates = {}
            for j in range(len(ids) - 1):
                pair = (ids[j], ids[j + 1])
                if pair in self.ranks:
                    candidates[pair] = self.ranks[pair]
            if not candidates:
                break
            pair = min(candidates, key=candidates.get)
            ids = list(merge_sequence(ids, pair, self.merges[pair]))
        self.cache[raw_bytes] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids

    def decode(self, ids):
        data = b""
        for i in ids:
            data += self.itos.get(i, b"")
        return data.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self.vocab_size


def train_tokenizer(train_text, vocab_size):
    tokenizer = BPETokenizer()
    tokenizer.train(train_text, vocab_size)
    return tokenizer

In [ ]:
# 4b · Import your modules + a tiny save/load helper for the tokenizer
import sys, importlib
sys.path.insert(0, os.path.abspath("a3src"))
import rope, attention, model_arch, a1_tokenizer
importlib.reload(rope); importlib.reload(attention); importlib.reload(model_arch)
from a1_tokenizer import BPETokenizer, train_tokenizer
from model_arch import LMConfig, DecoderLM

def save_tokenizer(tok, path):
    cache = tok.cache; tok.cache = {}            # don't pickle the (large) encode cache
    try:
        with open(path, "wb") as f: pickle.dump(tok, f)
    finally:
        tok.cache = cache

def load_tokenizer(path):
    with open(path, "rb") as f: return pickle.load(f)

print("Imported your A3 DecoderLM + BPE tokenizer.")

## 5 · Load the legal corpus
The dataset is gated and the training files are large, so we **stream** them with `ijson` (low memory) and pull text up to a character budget. The extractor is defensive because the exact JSON schema isn't published — after this runs, **eyeball the printed sample** to confirm it grabbed real legal text.

In [ ]:
# 5 · Discover + stream the Phase-1 files ----------------------------------
from huggingface_hub import HfApi, hf_hub_download
import ijson

api = HfApi()
repo_files = api.list_repo_files(CONFIG["hf_repo"], repo_type="dataset")
phase_files = [f for f in repo_files if f.startswith(CONFIG["phase_dir"] + "/")]
print("Phase-1 files in the repo:")
for f in phase_files: print("  ", f)

def dl(fname):
    return hf_hub_download(CONFIG["hf_repo"], fname, repo_type="dataset")

train_paths = {}
for src in CONFIG["sources"]:
    m = [f for f in phase_files if "/Train/" in f and src in f.lower() and f.endswith(".json")]
    if m: train_paths[src] = dl(m[0])
test_files = [f for f in phase_files if "/Test/" in f and f.endswith(".json")]
assert train_paths, "Found no Phase-1 training files."
assert test_files, "Found no Phase-1 test file."
test_path = dl(test_files[0])
print("\nTrain sources found:", list(train_paths))
print("Test file:", test_files[0])

In [ ]:
# 5b · Robust text extraction (handles several possible JSON shapes) -------
def iter_texts(obj):
    """Yield strings from a record of unknown shape (dict/list/str)."""
    if isinstance(obj, str):
        if obj.strip(): yield obj
    elif isinstance(obj, dict):
        for k in ("text", "content", "body", "document", "judgment", "raw_text"):
            if isinstance(obj.get(k), str) and obj[k].strip():
                yield obj[k]; return
        for v in obj.values():
            yield from iter_texts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_texts(v)

def _toplevel_kind(path):
    with open(path, "rb") as f:
        while True:
            c = f.read(1)
            if not c: return None
            if c.strip(): return c            # b"{" or b"["

def stream_texts(path, max_chars):
    """Stream documents from a possibly-huge JSON file, up to max_chars total."""
    kind = _toplevel_kind(path); got = 0
    with open(path, "rb") as f:
        gen = ijson.kvitems(f, "") if kind == b"{" else ijson.items(f, "item")
        for item in gen:
            val = item[1] if kind == b"{" else item
            for t in iter_texts(val):
                yield t; got += len(t)
                if got >= max_chars: return

char_budget = CONFIG["tok_train_chars"] + (CONFIG["train_tokens"] + CONFIG["val_tokens"]) * 6
per_source  = max(1, char_budget // max(1, len(train_paths)))

texts = []
for src, p in train_paths.items():
    n0 = len(texts)
    for t in stream_texts(p, per_source):
        texts.append(t)
    print(f"{src}: +{len(texts) - n0} documents")
random.shuffle(texts)
total_chars = sum(len(t) for t in texts)
print(f"\nCollected {len(texts):,} documents, {total_chars:,} characters.")
print("\n--- SAMPLE (first 600 chars of one document) ---\n")
print(texts[0][:600])

In [ ]:
# 6 · Train (or load) your byte-level BPE tokenizer on the legal text ------
tok_path = os.path.join(CONFIG["out_dir"], "legal_bpe.pkl")
if os.path.exists(tok_path):
    tokenizer = load_tokenizer(tok_path)
    print("Loaded cached tokenizer.")
else:
    tok_text = ("\n\n".join(texts))[:CONFIG["tok_train_chars"]]
    print(f"Training BPE (vocab={CONFIG['vocab_size']}) on {len(tok_text):,} chars ... (one-time, can take a while)")
    t0 = time.time()
    tokenizer = train_tokenizer(tok_text, CONFIG["vocab_size"])
    save_tokenizer(tokenizer, tok_path)
    print(f"Done in {time.time()-t0:.0f}s. Saved to {tok_path}")

VOCAB = tokenizer.get_vocab_size()
PAD, UNK, BOS, EOS = 0, 1, 2, 3          # reserved ids in your A1 tokenizer
print("Vocab size:", VOCAB)
print("Round-trip check:", repr(tokenizer.decode(tokenizer.encode("In the Supreme Court of India,"))))

In [ ]:
# 6b · Encode the corpus into a stream of token ids ------------------------
need = CONFIG["train_tokens"] + CONFIG["val_tokens"]
ids = []
t0 = time.time()
for i, t in enumerate(texts):
    ids.extend(tokenizer.encode(t)); ids.append(EOS)
    if i % 200 == 0:
        print(f"  encoded {i:,} docs -> {len(ids):,} tokens ({time.time()-t0:.0f}s)")
    if len(ids) >= need: break
ids = ids[:need]
val_n = CONFIG["val_tokens"]
train_ids = torch.tensor(ids[:-val_n], dtype=torch.long)
val_ids   = torch.tensor(ids[-val_n:], dtype=torch.long)
print(f"Train tokens: {train_ids.numel():,} | Val tokens: {val_ids.numel():,}")

In [ ]:
# 7 · Build your DecoderLM and check the Phase-1 parameter budget ----------
cfg = LMConfig(
    vocab_size=VOCAB, d_model=CONFIG["d_model"], n_layers=CONFIG["n_layers"],
    n_heads=CONFIG["n_heads"], n_kv_heads=CONFIG["n_kv_heads"], d_ff=CONFIG["d_ff"],
    max_seq_len=CONFIG["max_seq_len"], dropout=CONFIG["dropout"],
    pos_emb_type=CONFIG["pos_emb_type"],
)
model = DecoderLM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
if not SMOKE_TEST:
    assert 50_000_000 <= n_params <= 300_000_000, \
        "Phase-1 model must be 50M-300M params. Adjust d_model / n_layers / vocab_size."
    print("Within the 50M-300M Phase-1 budget.")

In [ ]:
# 8 · Train from scratch ---------------------------------------------------
_use_cuda = (device == "cuda")
def amp_autocast():
    return torch.autocast(device_type=("cuda" if _use_cuda else "cpu"), enabled=_use_cuda)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=_use_cuda)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=_use_cuda)

def get_batch(data, bs, seqlen):
    ix = torch.randint(0, data.numel() - seqlen - 1, (bs,))
    x = torch.stack([data[i:i+seqlen]       for i in ix])
    y = torch.stack([data[i+1:i+seqlen+1]   for i in ix])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

def lr_at(step):
    w, T = CONFIG["warmup_steps"], CONFIG["max_steps"]
    if step < w: return CONFIG["lr"] * step / max(1, w)
    p = (step - w) / max(1, T - w)
    return 0.5 * CONFIG["lr"] * (1 + math.cos(math.pi * min(1.0, p)))

@torch.no_grad()
def evaluate(n=40):
    model.eval(); losses = []
    for _ in range(n):
        x, y = get_batch(val_ids, CONFIG["batch_size"], CONFIG["max_seq_len"])
        with amp_autocast():
            loss = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1))
        losses.append(loss.item())
    model.train()
    l = sum(losses) / len(losses)
    return l, math.exp(min(20, l))

opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                        weight_decay=CONFIG["weight_decay"], betas=(0.9, 0.95))
ckpt = os.path.join(CONFIG["out_dir"], "best_model.pt")
best = float("inf"); model.train(); t0 = time.time()
for step in range(1, CONFIG["max_steps"] + 1):
    for g in opt.param_groups: g["lr"] = lr_at(step)
    opt.zero_grad(set_to_none=True)
    for _ in range(CONFIG["grad_accum"]):
        x, y = get_batch(train_ids, CONFIG["batch_size"], CONFIG["max_seq_len"])
        with amp_autocast():
            loss = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1)) / CONFIG["grad_accum"]
        scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
    scaler.step(opt); scaler.update()
    if step % CONFIG["eval_every"] == 0 or step == CONFIG["max_steps"]:
        vl, vppl = evaluate()
        print(f"step {step:>5} | lr {lr_at(step):.2e} | val loss {vl:.3f} | val ppl {vppl:8.1f} | {time.time()-t0:.0f}s")
        if vl < best:
            best = vl
            torch.save({"model": model.state_dict(), "cfg": vars(cfg)}, ckpt)
            print("        ^ new best, checkpoint saved")
print("Training done. Best val loss:", round(best, 3))

In [ ]:
# 8b · Load the best checkpoint for generation
state = torch.load(ckpt, map_location=device)
model.load_state_dict(state["model"]); model.eval()
print("Loaded best checkpoint.")

In [ ]:
# 9 · Generation (temperature + top-k + top-p sampling, from scratch) ------
@torch.no_grad()
def generate(prompt, max_new_tokens, temperature, top_k, top_p):
    ids = tokenizer.encode(prompt)[-(CONFIG["max_seq_len"] - 1):]
    ctx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    out = []
    for _ in range(max_new_tokens):
        window = ctx[:, -CONFIG["max_seq_len"]:]
        with amp_autocast():
            logits = model(window)[:, -1, :].float()
        logits = logits / max(1e-6, temperature)
        if top_k:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
        probs = F.softmax(logits, dim=-1)
        if top_p and top_p < 1.0:
            sp, si = torch.sort(probs, descending=True)
            keep = (torch.cumsum(sp, dim=-1) - sp) <= top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
        else:
            nxt = torch.multinomial(probs, 1)
        tok = int(nxt)
        if tok == EOS: break
        ctx = torch.cat([ctx, nxt], dim=1); out.append(tok)
    return tokenizer.decode(out)

def first_n_words(text, n):
    return " ".join(text.split()[:n])

# quick look
demo = generate("In the Supreme Court of India, the appellant contended that",
                CONFIG["max_new_tokens"], CONFIG["temperature"], CONFIG["top_k"], CONFIG["top_p"])
print(first_n_words(demo, CONFIG["gen_words"]))

In [ ]:
# 10 · Generate a continuation for every official test prompt --------------
def get_text_field(v):
    if isinstance(v, str): return v
    if isinstance(v, dict):
        for k in ("text", "content", "prompt", "body"):
            if isinstance(v.get(k), str): return v[k]
        for vv in v.values():
            if isinstance(vv, str): return vv
    return ""

with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)
assert isinstance(test_data, dict), f"Expected test.json to be an id->record dict, got {type(test_data)}"

preds = {}
keys = list(test_data.keys())
t0 = time.time()
for j, k in enumerate(keys):
    prompt = get_text_field(test_data[k])
    cont = generate(prompt, CONFIG["max_new_tokens"], CONFIG["temperature"],
                    CONFIG["top_k"], CONFIG["top_p"])
    preds[k] = {"text": first_n_words(cont, CONFIG["gen_words"])}
    if j % 20 == 0:
        print(f"  {j+1}/{len(keys)} prompts done ({time.time()-t0:.0f}s)")
print(f"Generated {len(preds)} continuations.")

In [ ]:
# 11 · Validate + write prediction.json + zip submission.zip ---------------
# ---- sanity checks the Codabench evaluator will also do ----
assert set(preds.keys()) == set(test_data.keys()), "Test IDs must match exactly (no missing/extra)."
for k, v in preds.items():
    assert isinstance(v["text"], str) and v["text"].strip(), f"Empty continuation for id {k}"
wc = [len(v["text"].split()) for v in preds.values()]
print(f"Word counts: min {min(wc)}, max {max(wc)}, mean {sum(wc)/len(wc):.1f} (target 128)")

pred_path = os.path.join(CONFIG["out_dir"], "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)

zip_path = os.path.join(CONFIG["out_dir"], "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")     # MUST be at the zip root

print("Wrote:", pred_path)
print("Wrote:", zip_path, "-> upload this to Codabench.")

## Done — what to do next
- Download **`submission.zip`** from the Kaggle output panel and upload it on Codabench.
- **Remember:** starter-style / smoke-test runs don't count toward your minimum 4. Set `SMOKE_TEST = False`, train a real model, and submit that.

### Building your 4 required architectures (next step)
This notebook is **architecture #1: a decoder-only Transformer with MHA + sinusoidal positions** (your A3 baseline, scaled up). The other three should be *meaningfully different* designs, e.g.:
- **#2 — GQA + RoPE**: set `n_kv_heads` well below `n_heads` and `pos_emb_type="rope"` (both already in your A3 code).
- **#3 — MLA (multi-head latent attention)**: you implemented this in A3 Q2 — we'll wire in that attention variant.
- **#4 — a genuinely different design**: e.g. an encoder-decoder, or a lightweight/efficient Transformer.

We'll do these one at a time. Tell me when your first run works and we'll move on.